In [7]:
!pip install pyspark

In [11]:
#load logs

from pyspark.sql import SparkSession

def create_spark_session():
    spark = SparkSession.builder.appName("LogAnomalyDetection").getOrCreate()
    return spark

def load_logs(spark, path):
    return spark.read.text(path)

In [12]:
#preprocess

from pyspark.sql.functions import regexp_extract

def preprocess_logs(df):
    df = df.withColumnRenamed("value", "log")

    df = df.withColumn(
        "level",
        regexp_extract(df["log"], "(INFO|ERROR|WARN|WARNING)", 1)
    )

    df = df.withColumn(
        "ip",
        regexp_extract(df["log"], r"(\d+\.\d+\.\d+\.\d+)", 1)
    )

    return df

In [13]:
#features

from pyspark.sql.functions import length, when
from pyspark.ml.feature import VectorAssembler

def create_features(df):
    df = df.withColumn("length", length(df["log"]))

    df = df.withColumn(
        "is_error",
        when(df["level"] == "ERROR", 1).otherwise(0)
    )

    df = df.withColumn(
        "is_warning",
        when((df["level"] == "WARN") | (df["level"] == "WARNING"), 1).otherwise(0)
    )

    assembler = VectorAssembler(
        inputCols=["length", "is_error", "is_warning"],
        outputCol="features"
    )

    return assembler.transform(df)

In [14]:
#model

from pyspark.ml.clustering import KMeans

def train_model(df):
    kmeans = KMeans(k=2, seed=42)
    return kmeans.fit(df)

def predict(model, df):
    return model.transform(df)

def show_anomalies(predictions):
    predictions.groupBy("prediction").count().show()

In [16]:
#main

spark = create_spark_session()

#path = "/content/HDFS_2k.log"
import os

if os.path.exists("/content"):
    path = "/content/*.log"           #for colab
else:
    path = "*.log"                    #for git

df = load_logs(spark, path)

df = preprocess_logs(df)

df = create_features(df)

model = train_model(df)

predictions = predict(model, df)

#  1: showing all predictions
predictions.show(10, truncate=False)

# 2: get cluster counts
counts = predictions.groupBy("prediction").count().collect()

#  3: find smaller cluster which is actual anomaly cluster
anomaly_cluster = min(counts, key=lambda x: x['count'])['prediction']

print("Anomaly cluster is:", anomaly_cluster)

# 4: show anomalies ONLY
predictions.filter(predictions.prediction == anomaly_cluster).show(10, False)

# 5: summary
show_anomalies(predictions)

spark.stop()

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+-------------+------+--------+----------+---------------+----------+
|log                                                                                                                                                              |level|ip           |length|is_error|is_warning|features       |prediction|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+-------------+------+--------+----------+---------------+----------+
|081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating                                               |INFO |             |114   |0       |0         |[114.0,0.0,0.0]|0         |
|081109 203807 222 INFO dfs.DataNode$PacketRespo